# counter-cube walkthrough

Shows how to use counter-cube at each level: Task → Tool → Pluggable tool.

In [1]:
from cube.core import Action
from counter_cube import CounterBenchmark, CounterTaskConfig, CounterToolConfig
from counter_cube.pluggable_tool import CounterToolConfig as DynConfig, CounterToolPluggable

## 1. Task — episode loop

In [2]:
benchmark = CounterBenchmark()
benchmark.setup()
task_configs = {c.task_id: c for c in benchmark.get_task_configs()}
print("Available tasks:", list(task_configs.keys()))

Benchmark setup did not define any runtime context. If your tasks require shared infrastructure, please ensure that self._runtime_context is populated.
Benchmark initialization did not define a container backend.
Benchmark initialization did not define a default tool config.
Benchmark initialization did not define a seed generator.


Available tasks: ['count-to-3', 'count-to-3-with-decrement', 'count-by-2']


In [3]:
task = task_configs["count-to-3"].make()
obs, info = task.reset()
print("Target:", info["target"])

while True:
    env_out = task.step(Action(name="increment", arguments={}))
    print(env_out.obs.contents[0].data, f"| reward={env_out.reward} done={env_out.done}")
    if env_out.done:
        break

task.close()

Target: 3
Counter value is: 1 | reward=0.0 done=False
Counter value is: 2 | reward=0.0 done=False
Counter value is: 3 | reward=1.0 done=True


In [4]:
# Tool config can be overridden per-task
task = CounterTaskConfig(
    task_id="count-to-3",
    tool_config=CounterToolConfig(enable_decrement=True),
).make()
task.reset()
print("Actions:", [a.name for a in task.action_set])
task.close()

Actions: ['decrement', 'get_value', 'increment']


## 2. Tool — execute actions directly

In [5]:
# Default config: only increment and get_value are exposed
tool = CounterToolConfig().make()
print("Actions:", [a.name for a in tool.action_set])

print(tool.execute_action(Action(name="increment", arguments={})).contents[0].data)
print(tool.execute_action(Action(name="get_value", arguments={})).contents[0].data)

Actions: ['get_value', 'increment']
Counter value is: 1
Counter value is: 1


In [6]:
# Enable optional actions via config flags
tool = CounterToolConfig(enable_decrement=True, enable_increment_by=True).make()
print("Actions:", [a.name for a in tool.action_set])

tool.execute_action(Action(name="increment_by", arguments={"value": 5}))
tool.execute_action(Action(name="decrement", arguments={}))
print(tool.execute_action(Action(name="get_value", arguments={})).contents[0].data)  # 4

Actions: ['decrement', 'get_value', 'increment', 'increment_by']
Counter value is: 4


In [7]:
# reset() restores the tool to its initial state
print("Before reset:", tool._env.counter)
tool.reset()
print("After reset:", tool._env.counter)

Before reset: 4
After reset: 0


## 3. Pluggable tool — add custom actions at runtime

In [8]:
tool = CounterToolPluggable(DynConfig())
print("Default actions:", [a.name for a in tool.action_set])


def reset_counter(env) -> str:
    """Reset counter to zero."""
    env.counter = 0
    return "Counter reset"


tool.add_tool_action(reset_counter)
print("After plug-in:", [a.name for a in tool.action_set])

tool.execute_action(Action(name="increment", arguments={}))
tool.execute_action(Action(name="increment", arguments={}))
print("Before:", tool._env.counter)

result = tool.execute_action(Action(name="reset_counter", arguments={}))
print(result.contents[0].data)
print("After:", tool._env.counter)

Default actions: ['increment']
After plug-in: ['increment', 'reset_counter']
Before: 2
Counter reset
After: 0
